# LUPE 2.0 Analysis Package: Google Colab Setup

Run this notebook **once** (and again whenever you want to update the package). It:

1. Connects your Google Drive
2. Downloads the LUPE analysis package into your Drive
3. Copies the analysis notebooks into `notebooks/`, ready to open
4. Downloads the LUPE behavior model (first run only) and checks that it loads

Everything is kept in your Drive, so your projects, results and figures are still there next time.

Press SHIFT + ENTER to run each cell.

### USER INPUT!

In [ ]:
## Folder in your Google Drive where the package will live (the analysis notebooks expect this same folder)
LUPE_DIR = '/content/drive/MyDrive/LUPE-2.0-NotebookAnalysisPackage'

## Where to download the package from (leave as is unless the Corder Lab tells you otherwise)
REPO_URL = 'https://github.com/justin05423/LUPE-2.0-NotebookAnalysisPackage.git'
BRANCH = 'main'

### Step 1: Connect Google Drive

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not running in Google Colab: skipping Drive (using LUPE_DIR as a normal folder).')

### Step 2: Download the package into your Drive

The first time, this downloads everything. If the package is already in your Drive, it tries to fetch updates.
Your projects in `processed_dataset/` are never touched. If an update would change a file you edited
(for example `utils/meta.py`), nothing is changed and your existing copy keeps working.

In [ ]:
def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout + result.stderr)
    return result.returncode

if not os.path.isdir(os.path.join(LUPE_DIR, '.git')):
    os.makedirs(os.path.dirname(LUPE_DIR), exist_ok=True)
    if run(['git', 'clone', '--branch', BRANCH, REPO_URL, LUPE_DIR]) != 0:
        raise RuntimeError('Download failed, see the message above.')
else:
    print(f'Package already in Drive at {LUPE_DIR}; checking for updates...')
    # --ff-only never merges: if an update would touch a file you changed (e.g. meta.py), git stops and changes nothing
    if run(['git', '-C', LUPE_DIR, 'pull', '--ff-only', 'origin', BRANCH]) != 0:
        print('Could not update automatically (see message above). Your existing copy still works.')

### Step 3: Put the analysis notebooks in `notebooks/`

Copies each notebook from `notebooks/template_copy-files-to-notebooks/` into `notebooks/`.
Notebooks you already have there are **not** overwritten (they may contain your settings).

In [ ]:
import shutil, glob
template_dir = os.path.join(LUPE_DIR, 'notebooks', 'template_copy-files-to-notebooks')
notebooks_dir = os.path.join(LUPE_DIR, 'notebooks')
for src in sorted(glob.glob(os.path.join(template_dir, '*.ipynb'))):
    dst = os.path.join(notebooks_dir, os.path.basename(src))
    if os.path.exists(dst):
        print('kept (already there):', os.path.basename(dst))
    else:
        shutil.copy(src, dst)
        print('copied:', os.path.basename(dst))

### Step 4: Check the behavior model

The first time, this downloads the classifier (`model/model_portable.npz`, 57 MB) from the LUPE-2.0-App GitHub release and checks its sha256. It works with the scikit-learn version Colab provides. After that it is already in your Drive and loads straight away.

In [ ]:
os.chdir(notebooks_dir)
if os.path.join(os.path.abspath(''), '../') not in sys.path:
    sys.path.append(os.path.join(os.path.abspath(''), '../'))
import sklearn
from utils.classification import load_model

model = load_model('../model/model.pkl')
print(f'Model ready: {len(model.estimators_)} trees, {model.n_features_in_} features, '
      f'{len(model.classes_)} behaviors (scikit-learn {sklearn.__version__})')

# SETUP COMPLETE!

**Next steps**

1. Upload your DeepLabCut CSV folders to Google Drive, organized as `<project>/<group>/<condition>/*.csv`.
2. In Google Drive, open `LUPE-2.0-NotebookAnalysisPackage/notebooks/` and double-click
   `1_preprocess_data.ipynb` (choose *Open with > Google Colaboratory*).
3. In every notebook, run the **Google Colab only** cell first, then continue as usual.
   Steps 1 to 3 (`1_preprocess_data`, `2_preprocess_get_features`, `3_preprocess_get_behaviors`) prepare a project;
   the `behavior_*` notebooks analyze it. Results are saved in `processed_dataset/<project>/figures/` in your Drive.